In [1]:
import sys, os
sys.path.insert(0, "/home/ubuntu/xenaConvert")
import xenaConvert
import scanpy as sc
import pandas as pd

In [2]:
tenXDataDir = "/mnt/efsCollisson/16-080L/scRNAseq/FH_16-080_Liver_Met_1"
count_file = "filtered_feature_bc_matrix.h5"
outputdir = "/mnt/efsCollisson/16-080L/scRNAseq/FH_16-080_Liver_Met_1/xena"
studyName = "16-080L scRNA-seq"
assay = "10x Chromium"
normalization = True

# for checking the data quickly

In [ ]:
adata = sc.read_10x_h5(os.path.join(tenXDataDir, count_file))
adata

# skip this section - mostly for debugging or reanalysis

In [ ]:
adata = xenaConvert.basic_analysis(adata)

In [5]:
adata

AnnData object with n_obs × n_vars = 8997 × 36601
    obs: 'n_genes', 'n_counts', 'louvain', 'leiden'
    var: 'gene_ids', 'feature_types', 'genome', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg', 'pca', 'neighbors', 'louvain', 'leiden', 'louvain_DE', 'leiden_DE'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'

In [7]:
# build maps and the associated matadata
xenaConvert.adataToMap(adata, outputdir, studyName)

# build cluster and associated metadata
xenaConvert.adataToCluster(adata, outputdir, studyName)  

unrecognized or ignored map: X_pca


## DE analysis
https://scanpy.readthedocs.io/en/stable/tutorials/basics/clustering.html#differentially-expressed-genes-as-markers

In [ ]:
adata.var_names_make_unique()

In [187]:
# differentially expressed genes for each cluster
sc.tl.rank_genes_groups(adata, groupby="louvain", key_added = "louvain_DE", mask_vars=adata.var.highly_variable, method="wilcoxon")
#sc.tl.rank_genes_groups(adata, groupby="louvain", mask_vars=adata.var.highly_variable, method="t-test")

In [196]:
result = adata.uns['louvain_DE']
clusters = result['names'].dtype.names
dic = {cluster: list(result['names'][cluster][:20]) for cluster in clusters}

In [197]:
import pprint
pprint.pp(dic, width=1000, indent=2)

{ '0': ['NDRG1', 'HSP90AA1', 'FAM13A', 'PGK1', 'WSB1', 'ENO1', 'DNAJB1', 'HSPB1', 'HSPH1', 'HSPE1', 'PFKFB4', 'IGFBP2', 'PTGES3', 'MALAT1', 'HSPD1', 'CCNY', 'GPI', 'OGT', 'LDHA', 'FLNB'],
  '1': ['NELL1', 'ZMAT4', 'NRXN3', 'KCNB2', 'TMEM132D', 'KCNQ5', 'SERGEF', 'FRAS1', 'PTPRN2', 'DGKB', 'ANK2', 'CFAP299', 'ADGRB3', 'SH3GL2', 'LSAMP', 'PLXNA2', 'NKAIN2', 'DLGAP1', 'RGS7', 'AC092691.1'],
  '2': ['PLCG2', 'MTRNR2L12', 'CERS4', 'NXN', 'SYNE2', 'AL627171.2', 'STOX2', 'SOX4', 'KIAA1217', 'ANKRD26', 'MUC4', 'SIPA1L3', 'SFTPB', 'MTRNR2L8', 'MUC16', 'KAZN', 'ASCL1', 'ATRX', 'DHRSX', 'DLGAP1'],
  '3': ['APOLD1', 'TOP2A', 'MELK', 'KNL1', 'SDK1', 'CENPI', 'ASPM', 'LINC01572', 'RRM2', 'MIR924HG', 'KIF15', 'KIF4A', 'CENPP', 'NUSAP1', 'SGO2', 'SMC4', 'MIS18BP1', 'C21orf58', 'PPM1E', 'CIT'],
  '4': ['MEG3', 'LINC02476', 'AL691420.1', 'MEIS2', 'SCHLAP1', 'MEG8', 'AC093515.1', 'LINC02484', 'LINC02163', 'CACNA2D1', 'ADGRL3', 'CADM1', 'CDH12', 'MARCH1', 'AC106799.2', 'LINC01194', 'DPP6', 'SEZ6L', 'NCAM1

In [198]:
import json
with open(os.path.join(outputdir, "louvain_wilcoxon.json"), "w") as outfile: 
    json.dump(dic, outfile)

# Single function to generate the whole thing

In [ ]:
adata = xenaConvert.tenXToXenaCountMatrix (tenXDataDir, outputdir, studyName, assay, normalization)

In [5]:
adata

AnnData object with n_obs × n_vars = 8997 × 36601
    obs: 'n_genes', 'n_counts', 'louvain', 'leiden'
    var: 'gene_ids', 'feature_types', 'genome', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg', 'pca', 'neighbors', 'louvain', 'leiden'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'